# 02 — Grid the radar picks onto the Elmer DEM lattice

Collapse the ~771k full-density bed picks onto the **exact grid geometry of
the PNAS Elmer DEMs** — 150 m node spacing in the along-flow frame (the sif
grid is 166×166 nodes over 0…24750 m) — extended with the same registration
to cover the 50 km AOI (334×334 nodes, `LX,LY ∈ [−24975, +24975]`). The Elmer
domain is exactly the central 166×166 sub-lattice (indices 84…249), so any
product on this grid drops onto the sif DEM grid with zero resampling.

**Aggregation** (cells with multiple picks — multiple years, or dense
along-track sampling within one pass):
- **Ice thickness → harmonic mean** over *all* picks in the cell. `THICK` is
  the direct radar observable and is datum-free, so the datum-suspect seasons
  (2001/2003/2005) contribute here too. (The harmonic mean is applied to
  thickness rather than to bed *elevation* because elevation crosses zero in
  this AOI, where a harmonic mean is undefined — thickness is strictly
  positive. Swap the aggregator in `hmean` below if you want it otherwise.)
- **Surface elevation → arithmetic mean** over datum-good picks.
- **Bed (the conditioning value): `BED_HM = MEAN_SURF − HM_THICK`** — WGS-84
  ellipsoidal, like everything upstream; the geoid shift happens at the
  detrending step.
- `BED_MEAN`/`BED_STD` (arithmetic, datum-good only) ride along for QC — the
  within-cell std is a first look at the nugget.

This gridded product is the **"raw" data that MGSIM will interpolate** in
notebook 03.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DERIVED = Path('../data/derived')
FIGS = Path('../figures'); FIGS.mkdir(exist_ok=True)

picks = pd.read_csv(DERIVED / 'radar_picks_cw1951_aoi50km.csv')
aoi = json.load(open(DERIVED / 'aoi_cw1951.json'))

# Lattice: continue the sif DEM node lattice (0..24750 step 150 in DEM
# coords = -12375..12375 in LX/LY) outward to cover the AOI.
STEP = 150.0
NODE0 = -24975.0            # = -12375 - 84*150: stays on the DEM lattice
N_NODES = 334
DEM_I0, DEM_I1 = 84, 249    # the Elmer 166x166 sub-lattice
nodes = NODE0 + STEP * np.arange(N_NODES)
assert nodes[DEM_I0] == -12375.0 and nodes[DEM_I1] == 12375.0

ix = np.clip(np.rint((picks.LX.values - NODE0) / STEP).astype(int), 0, N_NODES - 1)
iy = np.clip(np.rint((picks.LY.values - NODE0) / STEP).astype(int), 0, N_NODES - 1)
picks = picks.assign(IX=ix, IY=iy)
print(f"{len(picks)} picks → binning to {N_NODES}×{N_NODES} nodes at {STEP:.0f} m")

In [ ]:
def hmean(v):
    v = np.asarray(v, float)
    return len(v) / np.sum(1.0 / v)

good = ~picks.DATUM_SUSPECT
g_all = picks.groupby(['IY', 'IX'])
g_good = picks[good].groupby(['IY', 'IX'])

cells = pd.DataFrame({
    'N_ALL': g_all.size(),
    'N_SEASONS': g_all.SEASON.nunique(),
    'HM_THICK': g_all.THICK.apply(hmean),
    'N_GOOD': g_good.size(),
    'MEAN_SURF': g_good.SURF_ELEV.mean(),
    'BED_MEAN': g_good.BED_ELEV.mean(),
    'BED_STD': g_good.BED_ELEV.std(),
})
cells['N_GOOD'] = cells.N_GOOD.fillna(0).astype(int)
cells['BED_HM'] = cells.MEAN_SURF - cells.HM_THICK
cells = cells.reset_index()
cells['LX'] = nodes[cells.IX]
cells['LY'] = nodes[cells.IY]
cells['DEM_X'] = cells.LX + 12375.0   # sif DEM coords (0..24750 inside domain)
cells['DEM_Y'] = cells.LY + 12375.0
cells['IN_DEMGRID'] = ((cells.IX >= DEM_I0) & (cells.IX <= DEM_I1)
                       & (cells.IY >= DEM_I0) & (cells.IY <= DEM_I1))

n_dem = int(cells.IN_DEMGRID.sum())
print(f"filled cells: {len(cells)} of {N_NODES**2} in the AOI "
      f"({100 * len(cells) / N_NODES**2:.1f}%)")
print(f"inside the Elmer DEM grid: {n_dem} of {166 * 166} "
      f"({100 * n_dem / 166**2:.1f}%) — conditioned nodes for MGSIM")
print(f"picks per filled cell: median {cells.N_ALL.median():.0f}, "
      f"max {cells.N_ALL.max()}")
print(f"cells with a BED_HM value: {int(cells.BED_HM.notna().sum())} "
      f"(needs ≥1 datum-good pick for the surface)")
print(f"within-cell BED_STD: median {cells.BED_STD.median():.1f} m "
      f"(multi-pick cells only) — a first nugget estimate")
cells.head()

In [ ]:
out_cols = ['IX', 'IY', 'LX', 'LY', 'DEM_X', 'DEM_Y', 'IN_DEMGRID',
            'N_ALL', 'N_GOOD', 'N_SEASONS', 'HM_THICK', 'MEAN_SURF',
            'BED_HM', 'BED_MEAN', 'BED_STD']
cells[out_cols].to_csv(DERIVED / 'gridded_bed_cw1951_150m.csv',
                       index=False, float_format='%.3f')

def to_grid(col):
    a = np.full((N_NODES, N_NODES), np.nan)
    a[cells.IY, cells.IX] = cells[col]
    return a

np.savez_compressed(
    DERIVED / 'gridded_bed_cw1951_150m.npz',
    nodes_l=nodes,
    bed_hm=to_grid('BED_HM'), bed_mean=to_grid('BED_MEAN'),
    bed_std=to_grid('BED_STD'), hm_thick=to_grid('HM_THICK'),
    mean_surf=to_grid('MEAN_SURF'),
    n_all=to_grid('N_ALL'), n_good=to_grid('N_GOOD'),
    meta=json.dumps({'step_m': STEP, 'node0_l': NODE0, 'n_nodes': N_NODES,
                     'dem_sub_lattice': [DEM_I0, DEM_I1],
                     'datum': 'WGS-84 ellipsoidal (shift ~-28 m to geoid)',
                     'bed_hm': 'MEAN_SURF (datum-good) - HM_THICK (all picks)'}))
print('wrote gridded_bed_cw1951_150m.{csv,npz}')

In [ ]:
# Gridded raw bed next to the published bed, plus the sampling density.
import xarray as xr
NC = Path('../../cinematic/source_data/pnas_sims/2019cw428_d25km_r2500m_csi6.5e6_tsic_viscous.nc')
TILE, ELMER = aoi['tile_half_m'], aoi['elmer_half_m']

def quad(half):
    return np.array([[-1, -1], [1, -1], [1, 1], [-1, 1], [-1, -1]]) * half / 1e3

bed_grid = to_grid('BED_HM') - 28.0   # ~geoid shift for display beside the nc
# Color range from the Elmer-domain cells — the AOI's NE corner holds a
# real ~-1300 m outlet trough that would otherwise flatten the scale.
dom = bed_grid[DEM_I0:DEM_I1 + 1, DEM_I0:DEM_I1 + 1]
vmin, vmax = np.nanpercentile(dom, [0.5, 99.5])
fig, axs = plt.subplots(1, 3, figsize=(17.5, 6.2), constrained_layout=True)
if NC.exists():
    ds = xr.open_dataset(NC)
    pub = ds.topo_bed.isel(case=0, level=0)
    axs[0].pcolormesh(ds.x / 1e3, ds.y / 1e3, pub, cmap='gist_earth',
                      vmin=vmin, vmax=vmax, shading='auto')
axs[0].set_title('published bed (analysis tile, geoid)')
ext = [nodes[0] / 1e3, nodes[-1] / 1e3] * 2
pm = axs[1].imshow(bed_grid, origin='lower', extent=ext, cmap='gist_earth',
                   vmin=vmin, vmax=vmax, interpolation='nearest')
axs[1].set_title('gridded radar bed BED_HM (−28 m, 150 m cells)')
fig.colorbar(pm, ax=axs[:2], shrink=0.85, label='bed elevation [m]')
nm = axs[2].imshow(to_grid('N_ALL'), origin='lower', extent=ext, cmap='magma',
                   norm=plt.matplotlib.colors.LogNorm(vmin=1), interpolation='nearest')
axs[2].set_title('picks per cell')
fig.colorbar(nm, ax=axs[2], shrink=0.85, label='N (log)')
for ax in axs:
    for half, ls in [(ELMER, '-'), (TILE, '--')]:
        q = quad(half)
        ax.plot(q[:, 0], q[:, 1], ls, color='k', lw=1.1)
    ax.set_aspect('equal')
    ax.set_xlim(nodes[0] / 1e3, nodes[-1] / 1e3)
    ax.set_ylim(nodes[0] / 1e3, nodes[-1] / 1e3)
    ax.set_xlabel('LX [km]')
axs[0].set_ylabel('LY [km]')
fig.savefig(FIGS / 'gridded_bed.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# Within-cell spread (repeat passes disagreeing inside one 150 m cell) and
# the gridded-vs-published residual inside the tile.
fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.4))
ms = cells[cells.N_GOOD > 3]
axs[0].hist(ms.BED_STD.dropna(), bins=50, color='#888',
            range=(0, np.nanpercentile(ms.BED_STD, 99)))
axs[0].axvline(ms.BED_STD.median(), color='k')
axs[0].set_xlabel('within-cell BED_ELEV std [m] (cells with >3 good picks)')
axs[0].set_title(f"median {ms.BED_STD.median():.1f} m — nugget-scale scatter")
if NC.exists():
    from scipy.interpolate import RegularGridInterpolator
    itp = RegularGridInterpolator((ds.y.values, ds.x.values), pub.values,
                                  bounds_error=False)
    t = cells[cells.IN_DEMGRID & cells.BED_HM.notna()].copy()
    t['PUB'] = itp(np.stack([t.LY, t.LX], 1))
    dz = (t.BED_HM - 28.0) - t.PUB
    dz = dz.dropna()
    axs[1].hist(dz, bins=60, color='#6b8dad',
                range=np.percentile(dz, [0.5, 99.5]))
    axs[1].axvline(0, color='r', ls='--')
    axs[1].axvline(dz.median(), color='k')
    axs[1].set_xlabel('gridded radar bed (−28 m) − published bed [m]')
    axs[1].set_title(f"n={len(dz)} cells · median {dz.median():+.1f} m · "
                     f"std {dz.std():.1f} m")
fig.tight_layout(); fig.savefig(FIGS / 'gridded_spread.png', dpi=160)
plt.show()
print('The right-hand spread is the field MGSIM will simulate: where the\n'
      'radar-constrained bed genuinely departs from the interpolated product.')

## Next

- **03 — variograms & realizations**: geoid shift, rescue of the
  datum-suspect surface via the published surface DEM where needed, detrend
  against the published bed, fit the residual variogram, and draw N≈10 seeded
  conditional MGSIM realizations on this same lattice (tapered to the
  published bed over the outer ~2 km of the Elmer domain).
- **04 — Elmer ensemble**: write each realization as a sif-format `.dat` DEM
  and run the published viscous setup per bed.